# Batch Normalization vs Layer Normalization

Example: a batch of 4 samples, each with 3 channels, at some spatial location.

---

## The Data Cube

Think of your data as a 3D cube:

```
        Channels (C)
        ──────────────►
       ┌─────┬─────┬─────┐
   B   │  s1 │  s1 │  s1 │  ← Sample 1
   a   ├─────┼─────┼─────┤
   t   │  s2 │  s2 │  s2 │  ← Sample 2
   c   ├─────┼─────┼─────┤
   h   │  s3 │  s3 │  s3 │  ← Sample 3
   │   ├─────┼─────┼─────┤
   ▼   │  s4 │  s4 │  s4 │  ← Sample 4
       └─────┴─────┴─────┘
         C1    C2    C3
```

---

## Batch Normalization — Normalize ACROSS the batch

For each channel, compute mean and variance **across all samples** in the batch:

```
        Channels (C)
        ──────────────►
       ┌─────┬─────┬─────┐
       │  s1 │  s1 │  s1 │  ▲
       ├─────┼─────┼─────┤  │
       │  s2 │  s2 │  s2 │  │  mean & variance
       ├─────┼─────┼─────┤  │  computed down
       │  s3 │  s3 │  s3 │  │  this direction
       ├─────┼─────┼─────┤  │
       │  s4 │  s4 │  s4 │  ▼
       └─────┴─────┴─────┘
         ↑     ↑     ↑
        μ1    μ2    μ3     ← one mean per channel
        σ1    σ2    σ3     ← one variance per channel
```

Each channel gets its own μ and σ computed from **all 4 samples**.

---

## Layer Normalization — Normalize ACROSS the channels

For each sample, compute mean and variance **across all channels** within that sample:

```
        Channels (C)
        ──────────────►
       ┌─────┬─────┬─────┐
       │  s1 │  s1 │  s1 │ → μ, σ computed across C1, C2, C3 for sample 1
       ├─────┼─────┼─────┤
       │  s2 │  s2 │  s2 │ → μ, σ computed across C1, C2, C3 for sample 2
       ├─────┼─────┼─────┤
       │  s3 │  s3 │  s3 │ → μ, σ computed across C1, C2, C3 for sample 3
       ├─────┼─────┼─────┤
       │  s4 │  s4 │  s4 │ → μ, σ computed across C1, C2, C3 for sample 4
       └─────┴─────┴─────┘
```

Each sample gets its own μ and σ, independent of other samples.

---

## Side by Side Comparison

```
BATCH NORM                          LAYER NORM

┌────┬────┬────┐                   ┌────┬────┬────┐
│ █  │ █  │ █  │                   │ ████████████ │
│ █  │ █  │ █  │                   ├────┴────┴────┤
│ █  │ █  │ █  │                   │ ████████████ │
│ █  │ █  │ █  │                   ├─────────────┤
└────┴────┴────┘                   │ ████████████ │
  ↑    ↑    ↑                      ├─────────────┤
normalize down                     │ ████████████ │
(across batch)                     └─────────────┘
                                   normalize across →
                                   (across channels)
```

---

## Key Practical Differences

| | Batch Norm | Layer Norm |
|--|-----------|------------|
| Normalizes over | Batch dimension | Channel/feature dimension |
| Depends on batch size | Yes — needs large batch | No — works with batch size 1 |
| Used in | CNNs, ResNets | Transformers, RNNs, LLMs |
| At inference | Uses running mean/variance | Uses same sample stats |
| Sequence tasks | Poor — sequences vary in length | Great — each token normalized independently |

---

## Why Transformers Use Layer Norm

In a transformer, each token in a sequence is one "sample." You can't batch norm across tokens because:

- Sequences have **variable lengths**
- Token positions are **not equivalent** across samples
- Batch sizes can be small

Layer norm normalizes each token's features independently — perfect for sequential data where each position should be treated on its own terms.

---

## Where to put Normalization Steps?

Putting Batch Normalization (BN) before activation (Linear -&gt; BN -&gt; ReLU) is the original, generally recommended practice to ensure inputs to the non-linearity are Gaussian-distributed. Placing it after (Linear -&gt; ReLU -&gt; BN) is sometimes used for specific architectures or to stabilize non-Gaussian outputs, but it may alter normalized activations. 

• BN Before Activation (Standard): 

	• Goal: Normalize the raw, linear output ($W_x + b$) before the nonlinearity is applied. 
	• Advantage: Centers the data around the active region of the activation function (like sigmoid/tanh) or prevents "dying ReLU" issues by keeping data centered around zero before clipping. 
	• Result: More stable, faster training. 

• BN After Activation: 

	• Goal: Normalize the non-linear outputs (e.g., after ReLU). 
	• Advantage: Some empirical evidence suggests it may perform slightly better in specific deep residual networks (e.g., ResNet). 
	• Disadvantage: ReLU mapping all negative values to zero can create a skewed distribution (mean not zero).